In [1]:
import os

from dotenv import load_dotenv

from langchain_community.docstore.document import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever

from smolagents import CodeAgent, LiteLLMModel, Tool

load_dotenv()

/var/folders/nm/k95fxnfj0l73zhz906wzyhlc0000gn/T/ipykernel_4667/881724116.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.docstore.document import Document


True

In [2]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "groq/openai/gpt-oss-120b")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

model = LiteLLMModel(
    model_id=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0.2,
    max_tokens=4096,
)

print(f"Model configured: {GROQ_MODEL}")

Model configured: groq/openai/gpt-oss-120b


In [3]:
party_ideas = [
    {
        "text": "A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.",
        "source": "Party Ideas 1"
    },
    {
        "text": "Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.",
        "source": "Entertainment Ideas"
    },
    {
        "text": "For catering, serve dishes named after superheroes, like The Hulk's Green Smoothie and Iron Man's Power Steak.",
        "source": "Catering Ideas"
    },
    {
        "text": "Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.",
        "source": "Decoration Ideas"
    },
    {
        "text": "Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.",
        "source": "Entertainment Ideas"
    }
]

print(f"Knowledge base contains {len(party_ideas)} documents.")

Knowledge base contains 5 documents.


In [4]:
source_docs = [
    Document(
        page_content=doc["text"],
        metadata={"source": doc["source"]}
    )
    for doc in party_ideas
]

print(source_docs[0])

page_content='A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.' metadata={'source': 'Party Ideas 1'}


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True,
    strip_whitespace=True,
    separators=["\n\n", "\n", ".", " ", ""],
)

docs_processed = text_splitter.split_documents(source_docs)

print(f"Original documents: {len(source_docs)}")
print(f"Processed chunks: {len(docs_processed)}")

Original documents: 5
Processed chunks: 5


In [6]:
class PartyPlanningRetrieverTool(Tool):

    name = "party_planning_retriever"

    description = (
        "Retrieves relevant party planning ideas for Alfred's "
        "superhero-themed party at Wayne Manor. "
        "Use this tool to find information about decorations, "
        "entertainment, catering, and luxury party arrangements."
    )

    inputs = {
        "query": {
            "type": "string",
            "description": (
                "A search query related to superhero party planning."
            ),
        }
    }

    output_type = "string"

    def __init__(self, docs, **kwargs):
        super().__init__(**kwargs)

        self.retriever = BM25Retriever.from_documents(
            docs,
            k=5
        )

    def forward(self, query: str) -> str:

        assert isinstance(query, str), (
            "Your search query must be a string."
        )

        docs = self.retriever.invoke(query)

        if not docs:
            return "No relevant party planning ideas found."

        results = []

        for i, doc in enumerate(docs, start=1):

            source = doc.metadata.get("source", "Unknown")

            results.append(
                f"""
===== Idea {i} =====

Source: {source}

Content: {doc.page_content}
"""
            )

        return "\n".join(results)

In [8]:
party_planning_retriever = PartyPlanningRetrieverTool(
    docs_processed
)

print("Party planning retriever initialized successfully.")

Party planning retriever initialized successfully.


In [9]:
retrieved_results = party_planning_retriever.forward(
    "Luxury superhero party decorations"
)

print(retrieved_results)


===== Idea 1 =====

Source: Decoration Ideas

Content: Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.


===== Idea 2 =====

Source: Entertainment Ideas

Content: Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.


===== Idea 3 =====

Source: Catering Ideas

Content: For catering, serve dishes named after superheroes, like The Hulk's Green Smoothie and Iron Man's Power Steak.


===== Idea 4 =====

Source: Entertainment Ideas

Content: Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.


===== Idea 5 =====

Source: Party Ideas 1

Content: A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.



In [11]:
agent = CodeAgent(
    model=model,
    tools=[
        party_planning_retriever
    ],
    max_steps=2,
    verbosity_level=1,
)

In [12]:
response = agent.run(
    """
    Find ideas for a luxury superhero-themed party at Wayne Manor.

    Include:

    1. Luxury decorations and venue arrangements.
    2. Entertainment activities for guests.
    3. Creative superhero-themed catering options.

    Use the party planning retrieval tool to gather relevant ideas.

    Organize your final answer into clear sections.
    """
)

print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find ideas for a luxury superhero-themed party at Wayne Manor.                                                  │
│                                                                                                                 │
│     Include:                                                                                                    │
│                                                                                                                 │
│     1. Luxury decorations and venue arrangements.                                                               │
│     2. Entertainment activities for guests.                                                                     │
│     3. Creative superhero-themed catering options.                                                              │
│                                                                                                                 │
│     Use the party planning retrieval tool to gather relevant ideas.                                             │
│                                                                                                                 │
│     Organize your final answer into clear sections.                                                             │
│                                                                                                                 │
╰─ LiteLLMModel - groq/openai/gpt-oss-120b ───────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
                                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: None

[Step 1: Duration 1.74 seconds| Input tokens: 2,167 | Output tokens: 181]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (2 outputs left out)

[Step 2: Duration 1.36 seconds| Input tokens: 4,392 | Output tokens: 360]

Reached max steps.

[Step 3: Duration 4.96 seconds| Input tokens: 5,078 | Output tokens: 2,508]

## 🎉 Luxury Super‑Hero Party at Wayne Manor  
*An upscale, Gotham‑glamorous celebration that blends the elegance of Wayne Manor with the excitement of a superhero universe.*

---

### 1️⃣ Luxury Decorations & Venue Arrangements  

| Element | Details & Tips | Why It Works at Wayne Manor |
|---------|----------------|----------------------------|
| **Grand Entrance** | • Red‑carpet runway flanked by towering **Batman‑style gargoyles** (black marble, subtle LED eyes). <br>• A **“Welcome to the Bat‑Ball”** marquee in brushed‑gold lettering. | Sets a dramatic, cinematic tone the moment guests arrive. |
| **Super‑Hero Masquerade Ball Theme** | • Velvet drapes in deep midnight blue and rich burgundy, accented with **gold‑leaf trim**. <br>• Crystal chandeliers fitted with **soft, colored gels** that shift between iconic hero colors (red for Superman, teal for Aquaman, etc.). | Marries the classic Wayne Manor ballroom elegance with a playful, costume‑friendly vibe. |
| **Iconic Logos & Project